# 02 — RBM Baseline

**CineMind Phase 1 · Step 2** — Reproduce the reference repo's Restricted Boltzmann
Machine and measure its Precision@10 / Recall@10.

This gives us the **baseline** the two-tower model (notebook 04) must beat.

### What is an RBM?

Two layers of neurons.  The *visible* layer has one neuron per movie (on if the
user liked it).  The *hidden* layer holds latent "taste" units the model invents
on its own.  Training via **Contrastive Divergence (CD-1)** nudges connection
weights so the model can reconstruct a user's likes from those hidden tastes.
To recommend: feed in a user's known likes, read off which unseen movies light up.

**Requires:** `data/train_positives.csv`, `data/test_positives.csv` (from notebook 01)

In [1]:
import os, sys

# Run from project root regardless of where Jupyter was launched
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Make src/ importable
if not any('src' in p for p in sys.path):
    sys.path.insert(0, 'src')

import cinemind_utils as cu
print("cwd:", os.getcwd())

import numpy as np
import pandas as pd
import torch

torch.manual_seed(42)
np.random.seed(42)

cwd: c:\Users\shaur\Desktop\My codes shaurya\Content Recommendation system\cinemind _phase1\cinemind_phase1


## 1 · RBM class

In [2]:
class RBM:
    """Minimal Restricted Boltzmann Machine for collaborative filtering."""

    def __init__(self, n_visible, n_hidden=100, lr=0.1):
        self.W      = torch.randn(n_visible, n_hidden) * 0.01
        self.v_bias = torch.zeros(n_visible)
        self.h_bias = torch.zeros(n_hidden)
        self.lr     = lr

    def sample_h(self, v):
        prob_h = torch.sigmoid(v @ self.W + self.h_bias)
        return prob_h, torch.bernoulli(prob_h)

    def sample_v(self, h):
        prob_v = torch.sigmoid(h @ self.W.t() + self.v_bias)
        return prob_v, torch.bernoulli(prob_v)

    def train(self, data, epochs=15, batch_size=64):
        n = data.shape[0]
        for epoch in range(epochs):
            err  = 0.0
            perm = torch.randperm(n)
            for i in range(0, n - batch_size, batch_size):
                v0        = data[perm[i:i + batch_size]]
                ph0, h0   = self.sample_h(v0)       # positive phase
                pv1, v1   = self.sample_v(h0)        # reconstruct
                ph1, _    = self.sample_h(v1)        # negative phase
                self.W      += self.lr * (v0.t() @ ph0 - v1.t() @ ph1) / batch_size
                self.v_bias += self.lr * (v0 - v1).mean(0)
                self.h_bias += self.lr * (ph0 - ph1).mean(0)
                err += (v0 - v1).pow(2).mean().item()
            if (epoch + 1) % 5 == 0:
                print(f"  epoch {epoch+1:2d}  reconstruction error {err/(n//batch_size):.4f}")

    def recommend_scores(self, v):
        ph, _ = self.sample_h(v)
        pv, _ = self.sample_v(ph)
        return pv

## 2 · Load data and build the user–movie matrix

In [3]:
train = pd.read_csv("data/train_positives.csv")
test  = pd.read_csv("data/test_positives.csv")

n_items = int(max(train.movie_id.max(), test.movie_id.max())) + 1
n_users = int(max(train.user_id.max(), test.user_id.max())) + 1

# Binary interaction matrix: 1 if user liked movie in training set
matrix = torch.zeros(n_users, n_items)
for u, m in zip(train.user_id.values, train.movie_id.values):
    matrix[u, m] = 1.0

print(f"Matrix shape: {matrix.shape}  (density: {matrix.mean()*100:.2f}%)")

Matrix shape: torch.Size([944, 1675])  (density: 2.83%)


## 3 · Train

In [4]:
rbm = RBM(n_visible=n_items, n_hidden=100, lr=0.1)
rbm.train(matrix, epochs=15, batch_size=64)

  epoch  5  reconstruction error 0.0665
  epoch 10  reconstruction error 0.0496
  epoch 15  reconstruction error 0.0464


## 4 · Evaluate

In [5]:
seen    = train.groupby("user_id").movie_id.apply(set).to_dict()
truth   = test.groupby("user_id").movie_id.apply(set).to_dict()
top_pop = train.movie_id.value_counts().index.to_numpy()

def rbm_rank(u, k=10):
    scores = rbm.recommend_scores(matrix[u].unsqueeze(0)).squeeze(0).numpy()
    for m in seen.get(u, set()):
        scores[m] = -1e9      # never re-recommend something already seen
    return np.argpartition(-scores, k)[:k]

def pop_rank(u, k=10):
    s = seen.get(u, set())
    return [m for m in top_pop if m not in s][:k]

print("Evaluating ...")
p_pop, r_pop = cu.precision_recall_at_k(pop_rank, truth)
p_rbm, r_rbm = cu.precision_recall_at_k(rbm_rank, truth)

print(f"\n{'Model':<22}{'Precision@10':>14}{'Recall@10':>14}")
print("-" * 50)
print(f"{'Popularity baseline':<22}{p_pop:>14.4f}{r_pop:>14.4f}")
print(f"{'RBM':<22}{p_rbm:>14.4f}{r_rbm:>14.4f}")
print("-" * 50)
print("\nThese are the numbers notebook 04 (two-tower) aims to beat.")

Evaluating ...

Model                   Precision@10     Recall@10
--------------------------------------------------
Popularity baseline           0.0603        0.0665
RBM                           0.0745        0.0825
--------------------------------------------------

These are the numbers notebook 04 (two-tower) aims to beat.
